# AML-GNN — score a NEW dataset with the trained model (no retraining)

The Multi-PNA model trained on HI-Small is applied to transactions it has never seen, with its training-time feature encoding and alert threshold. If the data has labels, the report measures **out-of-dataset generalisation**.

1. **Runtime → Change runtime type → A100 GPU** (High-RAM). Any GPU works; CPU is fine for small files.
2. Upload `AML_GNN_model.zip` (code + trained model; create it with `python -m aml_gnn bundle model`, it appears in `dist/`) to `MyDrive/AML/`.
3. Choose the dataset in the settings cell:
   - `"HI-Medium"` — IBM AMLworld HI-Medium (32M transactions, a separate simulation), downloaded automatically; a date window keeps it to ~15 minutes.
   - `"custom"` — your own CSV in Drive plus a column mapping (copy and edit `examples/column_mapping.json`).
4. **Runtime → Run all.** The report, scores and a self-contained web app go to `MyDrive/AML/output/<name>/`.

In [ ]:
# ==== Settings ====
DRIVE_DIR = "/content/drive/MyDrive/AML"
DATASET   = "HI-Medium"                     # "HI-Medium" or "custom"

# HI-Medium spans 2022-09-01 → 2022-09-16; a window keeps memory and time small
START, END = "2022-09-01", "2022-09-03"

# custom: your CSV and column mapping in Drive (START/END also apply; set to None for all rows)
CUSTOM_CSV     = f"{DRIVE_DIR}/my_transactions.csv"
COLUMN_MAPPING = f"{DRIVE_DIR}/column_mapping.json"

NAME    = "hi-medium" if DATASET == "HI-Medium" else "my-bank"
BUNDLE  = f"{DRIVE_DIR}/AML_GNN_model.zip"
OUT_DIR = f"{DRIVE_DIR}/output/{NAME}"
PROJECT = "/content/AML-GNN"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib, os, torch
assert os.path.exists(BUNDLE), f"Not found: {BUNDLE}"
if DATASET == "custom":
    for p in (CUSTOM_CSV, COLUMN_MAPPING):
        assert os.path.exists(p), f"Not found: {p}"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)", "| torch", torch.__version__)

In [ ]:
# Code + trained model to local disk; install the two graph libraries
!rm -rf {PROJECT} && mkdir -p {PROJECT}
!unzip -q -o "{BUNDLE}" -d {PROJECT}
for p in ("aml_gnn/score.py", "artifacts/multipna_model.pt", "artifacts/feature_spec.json", "web/index.html"):
    assert os.path.exists(f"{PROJECT}/{p}"), f"Bundle is missing {p}"
!pip install -q torch_geometric torchmetrics

In [ ]:
# Get the data. HI-Medium comes from a public mirror and is checked against its published SHA-256.
if DATASET == "HI-Medium":
    BASE = "https://huggingface.co/datasets/OsamaMIT/IBM-AML-HI/resolve/main"
    !wget -q -O /content/HI-Medium_Trans.csv {BASE}/HI-Medium_Trans.csv
    !wget -q -O /content/HI-Medium_Patterns.txt {BASE}/HI-Medium_Patterns.txt
    sha = hashlib.sha256()
    with open("/content/HI-Medium_Trans.csv", "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            sha.update(block)
    assert sha.hexdigest() == "3126afb8155e7c8815d62bc5370549a7b5ae6bf7dfe872b3cd7813e66a3d7ff5", "download corrupted"
    INPUT, EXTRA = "/content/HI-Medium_Trans.csv", "--patterns /content/HI-Medium_Patterns.txt"
    print("HI-Medium downloaded and verified")
else:
    INPUT, EXTRA = CUSTOM_CSV, f"--columns '{COLUMN_MAPPING}'"
WINDOW = (f"--start {START} " if START else "") + (f"--end {END}" if END else "")

In [ ]:
# Score every transaction with the trained model, then build the report, cases and web app
!cd {PROJECT} && PYTHONUNBUFFERED=1 python -m aml_gnn score --input "{INPUT}" --name {NAME} {EXTRA} {WINDOW} 2>&1 | grep --line-buffered -v -E "FutureWarning|UserWarning|warnings.warn" | tee score.log

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{PROJECT}/runs/{NAME}/artifacts/results.md").read()))

In [ ]:
# Save the report, scores and the self-contained web app (serve or deploy the web/ folder) to Drive
os.makedirs(OUT_DIR, exist_ok=True)
!cp -r {PROJECT}/runs/{NAME}/artifacts "{OUT_DIR}/"
!cd {PROJECT}/runs/{NAME} && zip -qr "{OUT_DIR}/{NAME}_web.zip" web
!cp {PROJECT}/score.log "{OUT_DIR}/"
!ls -lh "{OUT_DIR}"